# Part 1 Alternative: Text Classification Representation Learning (BBC News Dataset)

In this alternative notebook, we download the standard BBC News Classification dataset via Pandas. We will train an MLP, snapshot the latent space embeddings across 11 stages (Epoch 0 through Epoch 10), and output them to a single Mantis-ready CSV file.


In [12]:
# Install required dependencies
!pip install torch torchvision pandas numpy scikit-learn umap-learn bokeh

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np
import urllib.request
import urllib.parse
import json
from sklearn.model_selection import train_test_split
import umap
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10
from bokeh.layouts import gridplot

output_notebook()


Loading BokehJS ...

## 1. Fetch the BBC News Dataset
We use Pandas to directly download the BBC News Classification CSV from a public GitHub repository. This dataset involves classifying news articles into business, entertainment, politics, sport, and tech.


In [14]:
# ---- Main Fetching ----
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
import umap
from bokeh.plotting import figure, show
from bokeh.layouts import gridplot
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10
from bokeh.io import output_notebook
output_notebook()

# Download BBC Dataset
url = "https://raw.githubusercontent.com/suraj-deshmukh/BBC-Dataset-News-Classification/master/dataset/dataset.csv"
print(f"Downloading BBC News Dataset from: {url}...")
df_data = pd.read_csv(url, encoding='latin1')

# The dataset has 'news' as the article and 'type' as the category label
# Let's map it to match our formatting
df_data = df_data.rename(columns={'news': 'Description', 'type': 'Category'})
# Produce pseudo titles by taking the first 40 chars of the article
df_data['Title'] = df_data['Description'].apply(lambda x: str(x)[:40] + "...")

topics = df_data['Category'].unique().tolist()
print(f"\nBBC Topics found: {topics}")
print(f"Fetched {len(df_data)} total articles!")

# Prepare TF-IDF features based exclusively on the BBC News article
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X = vectorizer.fit_transform(df_data['Description']).toarray()

# Map categories to indices
category_to_idx = {cat: idx for idx, cat in enumerate(topics)}
y = df_data['Category'].map(category_to_idx).values

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)


Loading BokehJS ...


BBC Topics found: ['business', 'entertainment', 'politics', 'sport', 'tech']
Fetched 2225 total articles!


## 2. Model Architecture
We track representations right before the ultimate `classifier` layer via the `self.embeddings` hook.


In [15]:
# We define a simple Multi-Layer Perceptron (MLP) for text classification.
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_classes):
        super(TextClassifier, self).__init__()
        self.fc1 = nn.Linear(vocab_size, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, hidden_dim) # Bottleneck representation
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.embeddings = None # This will act as our 'hook' to save the latent features
        
    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.fc2(out)
        self.embeddings = out.detach().cpu().numpy() # Save point
        out = self.relu(out)
        out = self.classifier(out)
        return out

model = TextClassifier(vocab_size=1000, hidden_dim=64, num_classes=len(topics))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)


## 3. Train & Capture Representations over 11 Epochs
Snapshot Epoch 0 (Initialization) and then do the standard gradient descent for 10 epochs, grabbing embeddings along the way.


In [16]:
epochs = 10
embeddings_per_epoch = []

# Epoch 0 (Initialization)
model.eval()
with torch.no_grad():
    model(X_tensor)
    # Save a copy of the latent representation to track how it evolves!
    embeddings_per_epoch.append(model.embeddings.copy())
model.train()

print("Training...")
# ---- Main Training Loop ----
# We iterate over the dataset multiple times (epochs)
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)
    loss.backward()
    optimizer.step()
    
    acc = (outputs.argmax(dim=1) == y_tensor).float().mean()
    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"Epoch [{(epoch+1):2d}/{epochs}], Loss: {loss.item():.4f}, Accuracy: {acc.item():.4f}")
    
    model.eval()
    with torch.no_grad():
        model(X_tensor)
        # Save a copy of the latent representation to track how it evolves!
        embeddings_per_epoch.append(model.embeddings.copy())
    model.train()

print(f"\nCaptured embeddings spanning {len(embeddings_per_epoch)} states.")


Training...
Epoch [ 1/10], Loss: 1.6087, Accuracy: 0.2297
Epoch [ 2/10], Loss: 1.5997, Accuracy: 0.2297
Epoch [ 4/10], Loss: 1.5756, Accuracy: 0.2297
Epoch [ 6/10], Loss: 1.5360, Accuracy: 0.2751
Epoch [ 8/10], Loss: 1.4760, Accuracy: 0.5901
Epoch [10/10], Loss: 1.3932, Accuracy: 0.8396

Captured embeddings spanning 11 states.


## 4. Export the Unified CSV
By formatting the output table with embedding_epoch_0, embedding_epoch_1, etc., Mantis can dynamically parse the multi-step training trajectory in one shot.


In [17]:
df_out = pd.DataFrame({
    'title': df_data['Title'],
    'categoric': df_data['Category'],
    'article_snippet': df_data['Description'].apply(lambda x: x[:150] + "...")
})

for i, embs in enumerate(embeddings_per_epoch):
    df_out[f'embedding_epoch_{i}'] = embs.tolist()

csv_name = 'text_analysis_bbc_mantis.csv'
df_out.to_csv(csv_name, index=False)
print(f"Saved {csv_name}!")


Saved text_analysis_bbc_mantis.csv!


## 5. Notebook Visualization grid (UMAP)
To visualize locally, we perform UMAP for all 11 epochs individually and place them in a time series grid. You should observe the data initially clustered randomly, then gradually organizing into structural islands!


In [18]:
from bokeh.io import output_notebook
output_notebook()
plots = []
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)

colors = Category10[max(3, len(topics))]
color_map = {cat: color for cat, color in zip(topics, colors)}
df_out['color'] = df_out['categoric'].map(color_map)

print("Generating UMAP plots. This will take a moment...")
for i, embs in enumerate(embeddings_per_epoch):
    umap_coords = reducer.fit_transform(embs)
    
    plot_df = pd.DataFrame({
        'coordinate1': umap_coords[:, 0],
        'coordinate2': umap_coords[:, 1],
        'color': df_out['color'],
        'categoric': df_out['categoric'],
        'title': df_out['title']
    })
    
    source = ColumnDataSource(plot_df)
    p = figure(title=f"Epoch {i}", width=300, height=300)
    p.scatter('coordinate1', 'coordinate2', size=4, color='color', legend_group='categoric', alpha=0.7, source=source)
    p.axis.visible = False
    if i != 0:
        p.legend.visible = False
        
    hover = HoverTool(tooltips=[("Category", "@categoric"), ("Title", "@title")])
    p.add_tools(hover)
    plots.append(p)

show(gridplot(plots, ncols=3))


Loading BokehJS ...

Generating UMAP plots. This will take a moment...


/Users/adityasengupta/miniconda3/envs/juliaenv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
